In [2]:
#!/usr/bin/env python
"""
Compute D_paths and H_norm (plus related descriptors) for all *stable* MuProMAC logs.

This script:
- Loads FIFO_EXP_stability_summary.csv
- Filters to rows with is_unstable == False and n_events > 0
- Iterates over these rows, loads each event log from "path"
- Calls compute_descriptors_for_file(...) to get D_paths, H_norm, entropies, etc.
- Writes a summary CSV with one row per log.
"""

import os
import pandas as pd
import numpy as np

RUN_TAG  = "251110"
OUT_ROOT = f"out/{RUN_TAG}"
RESULTS_DIR = os.path.join(OUT_ROOT, "results")
STAB_FILE = os.path.join(RESULTS_DIR, "FIFO_EXP_stability_summary.csv")

# Same warm-up fraction as before
WARMUP_FRAC = 0.17


def pop_var(x: pd.Series) -> float:
    x = x.to_numpy(dtype=float)
    if len(x) == 0:
        return np.nan
    mu = x.mean()
    return ((x - mu) ** 2).mean()


def _normalized_shannon(pi: np.ndarray) -> tuple[float, float, int]:
    """
    Return (H, H_norm, K) where:
      - H = Shannon entropy
      - H_norm = H / log(K)   (0 if K <= 1)
      - K = number of non-zero-probability states
    """
    pi = pi[pi > 0]
    K = len(pi)
    if K == 0:
        return 0.0, 0.0, 0
    H = -np.sum(pi * np.log(pi))
    if K <= 1:
        H_norm = 0.0
    else:
        H_norm = H / np.log(K)
    return float(H), float(H_norm), K


def _normalized_renyi2(pi: np.ndarray) -> tuple[float, float]:
    """
    Return (H2, H2_norm) where:
      - H2 = Rényi-2 entropy = -log(sum pi^2)
      - H2_norm = H2 / log(K)
    Uses K = number of non-zero-probability states.
    """
    pi = pi[pi > 0]
    K = len(pi)
    if K == 0:
        return 0.0, 0.0
    coll = np.sum(pi ** 2)        # collision probability
    H2 = -np.log(coll)            # Rényi-2
    if K <= 1:
        H2_norm = 0.0
    else:
        H2_norm = H2 / np.log(K)
    return float(H2), float(H2_norm)


def compute_descriptors_for_file(csv_path: str, warmup_frac: float = 0.17) -> dict:
    print(f"\n=== Processing {os.path.basename(csv_path)} ===")
    df = pd.read_csv(csv_path)

    # -------- warm-up selection on cases --------
    df_start = df[df["status"] == "START"][["case_id", "timestamp"]].copy()
    df_start = df_start.rename(columns={"timestamp": "start_time"})

    df_complete = df[(df["status"] == "COMPLETE") & (df["activity"] == "END")].copy()
    if df_complete.empty:
        raise ValueError(f"No COMPLETE END events in {csv_path}")

    df_complete = df_complete.merge(df_start, on="case_id", how="left")

    t_max = df["timestamp"].max()
    warmup_threshold = warmup_frac * t_max
    df_steady_complete = df_complete[df_complete["start_time"] >= warmup_threshold].copy()
    if df_steady_complete.empty:
        raise ValueError(f"No cases left after warm-up in {csv_path}")

    steady_case_ids = df_steady_complete["case_id"].unique()
    df_steady = df[df["case_id"].isin(steady_case_ids)].copy()

    # -------- overall cycle-time stats --------
    cycle_times = df_steady_complete["cycle_time"].to_numpy(dtype=float)
    mu_total = cycle_times.mean()
    var_total = ((cycle_times - mu_total) ** 2).mean()

    # -------- get running events --------
    df_run = df_steady[df_steady["status"] == "running"].copy()
    if df_run.empty:
        raise ValueError(f"No 'running' events for steady cases in {csv_path}")

    df_run = df_run.sort_values(["case_id", "timestamp", "activity", "resource"])

    # ======================================================================
    # MACHINE-LEVEL PATHS: Use resource column ONLY
    # ======================================================================
    def build_machine_path(group: pd.DataFrame) -> str:
        """Build path using ONLY resource identifiers."""
        nodes = []
        for _, row in group.iterrows():
            res = row.get("resource", None)
            if pd.notna(res) and isinstance(res, str) and res.strip() != "":
                nodes.append(res)
            else:
                # If resource is missing, we cannot build a valid machine path
                return None
        return ">".join(nodes) if nodes else None

    machine_paths = (
        df_run.groupby("case_id", group_keys=False)
              .apply(build_machine_path, include_groups=False)
              .reset_index(name="path")
    )
    
    # Remove cases where machine path could not be built
    machine_paths = machine_paths.dropna(subset=["path"]).copy()

    # ======================================================================
    # ACTIVITY-LEVEL PATHS: Use generic activity labels
    # ======================================================================
    # NEW HELPER FUNCTION to make activity names generic
    def normalize_activity_name(activity_str: str) -> str:
        """Removes the final _<number> index to generalize the activity."""
        # Split from the right once
        parts = activity_str.rsplit('_', 1)
        # If the last part is a number, we strip it.
        if len(parts) == 2 and parts[1].isdigit():
            return parts[0]
        # Otherwise, return the original string (e.g., 'V1_END')
        return activity_str

    def build_activity_path(group: pd.DataFrame) -> str:
        """Build path using ONLY GENERIC activity labels."""
        # *** MODIFIED LINE ***: Apply normalization before joining
        return ">".join(group["activity"].astype(str).apply(normalize_activity_name).tolist())

    activity_paths = (
        df_run.groupby("case_id", group_keys=False)
              .apply(build_activity_path, include_groups=False)
              .reset_index(name="act_path")
    )

    # The rest of the function (variance decomposition, entropy calculation) remains unchanged...
    # [TRUNCATED FOR BREVITY]

    # ======================================================================
    # MACHINE-LEVEL VARIANCE DECOMPOSITION AND ENTROPIES
    # ======================================================================
    case_level = df_steady_complete[["case_id", "cycle_time", "start_time"]].merge(
        machine_paths, on="case_id", how="left"
    )
    case_level = case_level.dropna(subset=["path"]).copy()
    N = len(case_level)
    
    if N == 0:
        print(f"  WARNING: No cases with valid machine-level paths")
        # Return NaN for all machine-level metrics
        D_paths = np.nan
        H_paths = H_norm_paths = H2_paths = H2_norm_paths = np.nan
        K_paths = 0
        Var_between = Var_within = np.nan
    else:
        # Variance decomposition by machine-level path
        path_stats = (
            case_level.groupby("path")["cycle_time"]
                      .agg(n="count", mean="mean", var=pop_var)
                      .reset_index()
        )

        N_check = path_stats["n"].sum()
        assert N_check == N

        weights = path_stats["n"] / N_check
        mu_from_paths = (path_stats["mean"] * path_stats["n"]).sum() / N_check

        Var_between = ((path_stats["mean"] - mu_from_paths) ** 2 * weights).sum()
        Var_within  = (path_stats["var"] * weights).sum()

        if var_total > 0:
            D_paths = Var_between / var_total
        else:
            D_paths = np.nan

        # Machine-level path entropies
        pi_paths = (path_stats["n"] / N_check).to_numpy(dtype=float)
        H_paths, H_norm_paths, K_paths = _normalized_shannon(pi_paths)
        H2_paths, H2_norm_paths = _normalized_renyi2(pi_paths)

    # ======================================================================
    # ACTIVITY-LEVEL PATH DISTRIBUTION AND ENTROPIES
    # ======================================================================
    case_level_act = (
        df_steady_complete[["case_id"]]
        .merge(activity_paths, on="case_id", how="left")
        .dropna(subset=["act_path"])
        .copy()
    )
    N_act = len(case_level_act)
    
    if N_act == 0:
        H_act = H_norm_act = H2_act = H2_norm_act = 0.0
        K_paths_act = 0
    else:
        act_stats = (
            case_level_act.groupby("act_path")["case_id"]
                          .agg(n="count")
                          .reset_index()
        )
        N_act_check = act_stats["n"].sum()
        assert N_act_check == N_act

        pi_act = (act_stats["n"] / N_act_check).to_numpy(dtype=float)
        H_act, H_norm_act, K_paths_act = _normalized_shannon(pi_act)
        H2_act, H2_norm_act = _normalized_renyi2(pi_act)

    print(f"  N_steady (with machine paths): {N}")
    print(f"  N_steady (with activity paths): {N_act}")
    print(f"  mu_total: {mu_total:.4f}, Var_total: {var_total:.4f}")
    if not np.isnan(D_paths):
        print(f"  Var_between: {Var_between:.4f}, Var_within: {Var_within:.4f}")
        print(f"  K_paths (machine): {K_paths}, D_paths: {D_paths:.4f}, "
              f"H_norm (machine): {H_norm_paths:.4f}, H2_norm (machine): {H2_norm_paths:.4f}")
    else:
        print(f"  K_paths (machine): {K_paths}, D_paths: NaN (no valid machine paths)")
    print(f"  K_paths_act (activity): {K_paths_act}, "
          f"H_norm_act: {H_norm_act:.4f}, H2_norm_act: {H2_norm_act:.4f}")

    return {
        "n_steady_cases": N,
        "mu_total": mu_total,
        "var_total": var_total,
        "var_between": Var_between,
        "var_within": Var_within,
        "var_bw_plus_within": Var_between + Var_within if not np.isnan(Var_between) else np.nan,
        "D_paths": D_paths,
        "H": H_paths,
        "H_norm": H_norm_paths,
        "K_paths": K_paths,
        "warmup_frac": warmup_frac,
        "warmup_threshold": warmup_threshold,
        "t_max": t_max,
        "H2_paths": H2_paths,
        "H2_norm_paths": H2_norm_paths,
        "N_act_cases": N_act,
        "K_paths_act": K_paths_act,
        "H_act": H_act,
        "H_norm_act": H_norm_act,
        "H2_act": H2_act,
        "H2_norm_act": H2_norm_act,
    }


def parse_log_name(log_name: str) -> dict:
    """
    Parse things like:
      FIFO_EXP_l0.28_pooled_C1_V6_A5_QC97_mild_all.csv
    into a dict of parameters.
    """
    base = os.path.basename(log_name)
    if base.endswith(".csv"):
        base = base[:-4]

    if base.startswith("FIFO_EXP_"):
        base = base[len("FIFO_EXP_"):]

    parts = base.split("_")
    params = {
        "log_name": log_name,
        "l": np.nan,
        "style": None,
        "count_preset": None,
        "variant_count": np.nan,
        "activity_total": np.nan,
        "qc_level": np.nan,
        "hetero": None,
    }
    try:
        if parts[0].startswith("l"):
            params["l"] = float(parts[0][1:])
        params["style"] = parts[1]
        params["count_preset"] = parts[2]
        if parts[3].startswith("V"):
            params["variant_count"] = int(parts[3][1:])
        if parts[4].startswith("A"):
            params["activity_total"] = int(parts[4][1:])
        if parts[5].startswith("QC"):
            params["qc_level"] = int(parts[5][2:]) / 100.0
        params["hetero"] = parts[6] if len(parts) > 6 else None
    except Exception as e:
        print(f"[WARN] Failed to parse log name: {log_name} ({e})")

    return params


def main():
    stab_df = pd.read_csv(STAB_FILE)

    # Clean column names – your CSV had 'is_unstable ' with a trailing space
    stab_df.columns = [c.strip() for c in stab_df.columns]

    if "is_unstable" not in stab_df.columns:
        raise ValueError(f"'is_unstable' not found in {STAB_FILE}")

    col = stab_df["is_unstable"]
    if col.dtype == bool:
        is_unstable_bool = col
    else:
        col_str = col.astype(str).str.strip().str.lower()
        is_unstable_bool = col_str.map(
            {"true": True, "false": False, "1": True, "0": False}
        )

    stab_df["is_unstable_bool"] = is_unstable_bool

    # Only stable logs with real events
    stable_df = stab_df[
        (stab_df["is_unstable_bool"] == False) &
        (stab_df["n_events"] > 0)
    ].copy()

    # Drop meta rows like FIFO_EXP_path_descriptors_* and FIFO_EXP_var_*
    stable_df = stable_df[
        ~stable_df["log_name"].str.startswith("FIFO_EXP_path_")
    ]
    stable_df = stable_df[
        ~stable_df["log_name"].str.startswith("FIFO_EXP_var_")
    ]

    print(f"Total logs in stability summary : {len(stab_df)}")
    print(f"Stable logs with events         : {len(stable_df)}")

    rows = []

    for _, row in stable_df.iterrows():
        log_name = row["log_name"]
        raw_path = row["path"]

        csv_path = os.path.normpath(raw_path)
        if not os.path.isfile(csv_path):
            print(f"WARNING: missing file, skipping: {csv_path}")
            continue

        # compute descriptors on the actual event log
        stats = compute_descriptors_for_file(csv_path, warmup_frac=WARMUP_FRAC)

        # parse parameters from log_name (style, qc_level, etc.)
        params = parse_log_name(log_name)

        # also keep some queue / WIP diagnostics from stability summary
        stats.update(params)
        stats.update({
            "n_events": row.get("n_events", np.nan),
            "n_cases": row.get("n_cases", np.nan),
            "wip_max": row.get("wip_max", np.nan),
            "q_max": row.get("q_max", np.nan),
            "is_unstable": row.get("is_unstable_bool", False),
        })

        rows.append(stats)

    summary_df = pd.DataFrame(rows)

    print("\n=== Path-based descriptor summary for STABLE logs ===")
    cols_to_show = [
        "l", "style", "hetero", "variant_count", "activity_total",
        "n_steady_cases", "mu_total", "var_total",
        "var_between", "var_within", "D_paths",
        "H_norm", "H2_norm_paths",
        "H_norm_act", "H2_norm_act",
        "K_paths", "K_paths_act"
    ]
    existing_cols = [c for c in cols_to_show if c in summary_df.columns]
    if existing_cols:
        print(summary_df[existing_cols])

    out_path = os.path.join(
        RESULTS_DIR,
        "FIFO_EXP_path_descriptors_machine_and_activity_level_STABLE.csv"
    )
    summary_df.to_csv(out_path, index=False)
    print(f"\nSaved summary to {out_path}")


if __name__ == "__main__":
    main()

Total logs in stability summary : 151
Stable logs with events         : 84

=== Processing FIFO_EXP_l0.28_dedicated_C1_V18_A5_QC97_identical.csv ===
  N_steady (with machine paths): 6937
  N_steady (with activity paths): 6937
  mu_total: 87.3268, Var_total: 2245.2584
  Var_between: 355.1556, Var_within: 1890.1027
  K_paths (machine): 729, D_paths: 0.1582, H_norm (machine): 0.8420, H2_norm (machine): 0.7979
  K_paths_act (activity): 729, H_norm_act: 0.8420, H2_norm_act: 0.7979

=== Processing FIFO_EXP_l0.28_dedicated_C1_V18_A5_QC97_mild_all.csv ===
  N_steady (with machine paths): 6927
  N_steady (with activity paths): 6927
  mu_total: 116.2135, Var_total: 4727.5036
  Var_between: 1533.2223, Var_within: 3194.2813
  K_paths (machine): 751, D_paths: 0.3243, H_norm (machine): 0.8431, H2_norm (machine): 0.7968
  K_paths_act (activity): 751, H_norm_act: 0.8431, H2_norm_act: 0.7968

=== Processing FIFO_EXP_l0.28_dedicated_C1_V18_A8_QC97_identical.csv ===
  N_steady (with machine paths): 6919
